# 📊 Cryptocurrency Technical Analysis\n## Professional-Grade Technical Analysis Following Industry Standards\n\nThis notebook provides comprehensive technical analysis including:\n1. **Price Action Analysis**: Candlestick patterns, support/resistance\n2. **Trend Indicators**: Moving averages, MACD, ADX\n3. **Momentum Indicators**: RSI, Stochastic, Williams %R\n4. **Volatility Indicators**: Bollinger Bands, ATR\n5. **Volume Analysis**: OBV, Volume Profile\n6. **Multi-timeframe Analysis**: Daily, 4H, 1H perspectives

## 1. Setup and Dependencies

In [ ]:
# Import required libraries with error handling\nimport sys\nimport os\nimport warnings\nfrom datetime import datetime, timedelta\nimport math\n\nwarnings.filterwarnings('ignore')\n\n# Add project path\nsys.path.append('/Users/aditya/PycharmProjects/PAlibaba_hackathon')\n\n# Install and import required packages\nrequired_packages = {\n    'pandas': 'pandas',\n    'numpy': 'numpy', \n    'matplotlib': 'matplotlib',\n    'seaborn': 'seaborn',\n    'plotly': 'plotly',\n    'ta': 'ta',  # Technical Analysis library\n    'yfinance': 'yfinance'  # Alternative data source\n}\n\nfor package_name, import_name in required_packages.items():\n    try:\n        if import_name == 'pandas':\n            import pandas as pd\n            print(f\"✅ {package_name} imported successfully\")\n        elif import_name == 'numpy':\n            import numpy as np\n            print(f\"✅ {package_name} imported successfully\")\n        elif import_name == 'matplotlib':\n            import matplotlib.pyplot as plt\n            import matplotlib.dates as mdates\n            from matplotlib.patches import Rectangle\n            print(f\"✅ {package_name} imported successfully\")\n        elif import_name == 'seaborn':\n            import seaborn as sns\n            print(f\"✅ {package_name} imported successfully\")\n        elif import_name == 'plotly':\n            import plotly.graph_objects as go\n            from plotly.subplots import make_subplots\n            import plotly.express as px\n            print(f\"✅ {package_name} imported successfully\")\n        elif import_name == 'ta':\n            import ta\n            from ta.trend import MACD, EMAIndicator, SMAIndicator, ADXIndicator\n            from ta.momentum import RSIIndicator, StochasticOscillator, WilliamsRIndicator\n            from ta.volatility import BollingerBands, AverageTrueRange\n            from ta.volume import OnBalanceVolumeIndicator, VolumeSMAIndicator\n            print(f\"✅ {package_name} imported successfully\")\n        elif import_name == 'yfinance':\n            import yfinance as yf\n            print(f\"✅ {package_name} imported successfully\")\n    except ImportError:\n        print(f\"❌ {package_name} not found. Installing...\")\n        !pip install {package_name}\n        # Re-import after installation\n        if import_name == 'pandas':\n            import pandas as pd\n        elif import_name == 'numpy':\n            import numpy as np\n        elif import_name == 'matplotlib':\n            import matplotlib.pyplot as plt\n            import matplotlib.dates as mdates\n            from matplotlib.patches import Rectangle\n        elif import_name == 'seaborn':\n            import seaborn as sns\n        elif import_name == 'plotly':\n            import plotly.graph_objects as go\n            from plotly.subplots import make_subplots\n            import plotly.express as px\n        elif import_name == 'ta':\n            import ta\n            from ta.trend import MACD, EMAIndicator, SMAIndicator, ADXIndicator\n            from ta.momentum import RSIIndicator, StochasticOscillator, WilliamsRIndicator\n            from ta.volatility import BollingerBands, AverageTrueRange\n            from ta.volume import OnBalanceVolumeIndicator, VolumeSMAIndicator\n        elif import_name == 'yfinance':\n            import yfinance as yf\n        print(f\"✅ {package_name} installed and imported successfully\")\n\n# Try to import custom modules\ntry:\n    from binance_api import binance_api\n    print(\"✅ Custom Binance API module imported\")\n    USE_BINANCE = True\nexcept ImportError:\n    print(\"⚠️ Custom Binance API not available, will use yfinance as fallback\")\n    USE_BINANCE = False\n\n# Configure plotting\nplt.style.use('default')\nplt.rcParams['figure.figsize'] = (12, 8)\nplt.rcParams['font.size'] = 10\n\n# Configure pandas\npd.set_option('display.max_columns', None)\npd.set_option('display.width', None)\npd.set_option('display.precision', 4)\n\nprint(\"\\n🎉 All dependencies loaded successfully!\")\nprint(f\"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\")

## 2. Configuration - Change These Settings

In [ ]:
# 🔧 CONFIGURATION SECTION - MODIFY THESE VALUES\nSYMBOL = 'BTC'      # Options: 'BTC', 'ETH', 'ADA', 'SOL', 'MATIC', etc.\nPERIOD = '3mo'      # Options: '1mo', '3mo', '6mo', '1y', '2y'\nINTERVAL = '1d'     # Options: '1d', '1h', '4h'\n\nprint(f\"🚀 Configuration Set:\")\nprint(f\"   Symbol: {SYMBOL}\")\nprint(f\"   Period: {PERIOD}\")\nprint(f\"   Interval: {INTERVAL}\")\nprint(\"\\n💡 To analyze a different cryptocurrency, change the SYMBOL variable above and re-run this cell.\")

## 3. Data Fetching and Technical Analysis

In [ ]:
def get_crypto_data(symbol, period='3mo', interval='1d'):\n    \"\"\"\n    Fetch cryptocurrency data from multiple sources with fallback.\n    \"\"\"\n    \n    # Try Binance API first if available\n    if USE_BINANCE and interval == '1d':\n        try:\n            print(f\"📊 Fetching {symbol} data from Binance API...\")\n            \n            # Convert period to days\n            period_days = {\n                '1mo': 30, '3mo': 90, '6mo': 180, \n                '1y': 365, '2y': 730\n            }.get(period, 90)\n            \n            result = binance_api.get_historical_data(symbol, interval='1d', days=period_days)\n            \n            if result.get('success'):\n                data_list = result['data']\n                df = pd.DataFrame(data_list)\n                \n                # Rename columns to standard format\n                df = df.rename(columns={\n                    'open': 'Open', 'high': 'High', 'low': 'Low', \n                    'close': 'Close', 'volume': 'Volume'\n                })\n                \n                df.set_index('timestamp', inplace=True)\n                df.index = pd.to_datetime(df.index)\n                \n                print(f\"✅ Successfully fetched {len(df)} records from Binance\")\n                return df\n                \n        except Exception as e:\n            print(f\"⚠️ Binance API failed: {e}\")\n    \n    # Fallback to yfinance\n    try:\n        print(f\"📊 Fetching {symbol} data from Yahoo Finance...\")\n        \n        # Convert symbol to Yahoo Finance format\n        yf_symbol = f\"{symbol}-USD\"\n        \n        # Fetch data\n        ticker = yf.Ticker(yf_symbol)\n        df = ticker.history(period=period, interval=interval)\n        \n        if df.empty:\n            raise ValueError(f\"No data found for {yf_symbol}\")\n        \n        # Ensure we have the required columns\n        required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']\n        for col in required_cols:\n            if col not in df.columns:\n                raise ValueError(f\"Missing required column: {col}\")\n        \n        print(f\"✅ Successfully fetched {len(df)} records from Yahoo Finance\")\n        return df[required_cols]\n        \n    except Exception as e:\n        print(f\"❌ Failed to fetch data: {e}\")\n        return None\n\ndef validate_data(df):\n    \"\"\"Validate and clean the data.\"\"\"\n    if df is None or df.empty:\n        return None\n    \n    # Remove any rows with NaN values\n    df = df.dropna()\n    \n    # Ensure positive values\n    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']\n    for col in numeric_cols:\n        if col in df.columns:\n            df[col] = pd.to_numeric(df[col], errors='coerce')\n            df = df[df[col] > 0]\n    \n    # Validate OHLC relationships\n    df = df[(df['High'] >= df['Low']) & \n            (df['High'] >= df['Open']) & \n            (df['High'] >= df['Close']) &\n            (df['Low'] <= df['Open']) & \n            (df['Low'] <= df['Close'])]\n    \n    return df.sort_index()\n\nprint(\"✅ Data fetching functions defined successfully!\")